In [1]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split, GridSearchCV
from tensorflow.keras.layers import Dense, Dropout, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from PIL import ImageFile
import joblib

In [3]:

# ==============================
# Step 1 — Load Dataset
# ==============================
print("Loading dataset...")

data_dir = "../../data sheets/training_set"
ImageFile.LOAD_TRUNCATED_IMAGES = True

batch_size = 32
img_size = (224, 224)

datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input,
    rotation_range=25,
    width_shift_range=0.25,
    height_shift_range=0.25,
    shear_range=0.2,
    zoom_range=0.3,
    brightness_range=[0.8, 1.2],
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset='training',
    shuffle=True
)

val_gen = datagen.flow_from_directory(
    data_dir,
    target_size=img_size,
    batch_size=batch_size,
    class_mode="sparse",
    subset='validation',
    shuffle=False
)

Loading dataset...
Found 4000 images belonging to 5 classes.
Found 1000 images belonging to 5 classes.


In [4]:
# ==============================
# Step 2 — Fine-tune ResNet50
# ==============================
print("Building ResNet50 base model...")

base_model = ResNet50(weights="imagenet", include_top=False, input_shape=(224, 224, 3))

# Freeze most layers, unfreeze last few
for layer in base_model.layers[:-20]:
    layer.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(512, activation='relu')(x)
x = Dropout(0.4)(x)
predictions = Dense(5, activation='softmax')(x)

ft_model = Model(inputs=base_model.input, outputs=predictions)
ft_model.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

print("Fine-tuning ResNet50...")
ft_model.fit(
    train_gen,
    validation_data=val_gen,
    epochs=5,
    callbacks=[early_stop],
    verbose=1
)


Building ResNet50 base model...
Fine-tuning ResNet50...


c:\Users\MODERN\anaconda3\Lib\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 695s 6s/step - accuracy: 0.2889 - loss: 1.7106 - val_accuracy: 0.4060 - val_loss: 1.4537
Epoch 2/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 487s 4s/step - accuracy: 0.4343 - loss: 1.3408 - val_accuracy: 0.4940 - val_loss: 1.2844
Epoch 3/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 527s 4s/step - accuracy: 0.5068 - loss: 1.2132 - val_accuracy: 0.4900 - val_loss: 1.2980
Epoch 4/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1425s 11s/step - accuracy: 0.5590 - loss: 1.1103 - val_accuracy: 0.5140 - val_loss: 1.2640
Epoch 5/5
125/125 ━━━━━━━━━━━━━━━━━━━━ 1509s 12s/step - accuracy: 0.6227 - loss: 0.9615 - val_accuracy: 0.5250 - val_loss: 1.2450


In [5]:

# ==============================
# Step 3 — Extract Features
# ==============================
print("Extracting features from fine-tuned ResNet50...")

feature_extractor = Model(inputs=ft_model.input, outputs=ft_model.get_layer("conv5_block3_out").output)

def extract_features(generator, model):
    features, labels = [], []
    for i in range(len(generator)):
        try:
            x_batch, y_batch = generator[i]
            feat_batch = model.predict(x_batch, verbose=0)
            feat_batch = feat_batch.reshape(feat_batch.shape[0], -1)
            features.append(feat_batch)
            labels.append(y_batch)
            if (i + 1) * batch_size >= generator.n:
                break
        except Exception as e:
            print(f"Skipping batch {i} due to error: {e}")
            continue
    X = np.vstack(features)
    y = np.hstack(labels)
    return X, y

X_train, y_train = extract_features(train_gen, feature_extractor)
X_test, y_test = extract_features(val_gen, feature_extractor)

print("Feature extraction complete.")
print("Train features:", X_train.shape, "Test features:", X_test.shape)


Extracting features from fine-tuned ResNet50...
Feature extraction complete.
Train features: (4000, 100352) Test features: (1000, 100352)


In [6]:

# ==============================
# Step 4 — Normalize + PCA
# ==============================
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

pca = PCA(n_components=256, random_state=42)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)

In [ ]:
# ==============================
# Step 5 — Train SVM
# ==============================
print("Performing Grid Search for best SVM parameters...")

param_grid = {
    'C': [1, 5, 10, 50],
    'gamma': ['scale', 0.001, 0.0005],
    'kernel': ['rbf']
}

grid = GridSearchCV(SVC(random_state=42, probability=True), param_grid, cv=3, n_jobs=-1, verbose=2)
grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)
clf = grid.best_estimator_

print("Training final SVM...")
clf.fit(X_train, y_train)

Performing Grid Search for best SVM parameters...
Fitting 3 folds for each of 12 candidates, totalling 36 fits
Best Parameters: {'C': 1, 'gamma': 'scale', 'kernel': 'rbf'}
Training final SVM...


SVC(C=1, random_state=42)

In [8]:

# ==============================
# Step 6 — Evaluate
# ==============================
print("\nEvaluating model...")
y_pred = clf.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)
print("\nAccuracy:", round(accuracy * 100, 2), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))



Evaluating model...

Accuracy: 52.0 %

Classification Report:
              precision    recall  f1-score   support

         0.0       0.51      0.50      0.50       200
         1.0       0.59      0.59      0.59       200
         2.0       0.36      0.40      0.38       200
         3.0       0.73      0.53      0.61       200
         4.0       0.50      0.58      0.54       200

    accuracy                           0.52      1000
   macro avg       0.54      0.52      0.52      1000
weighted avg       0.54      0.52      0.52      1000


Confusion Matrix:
[[100  27  43   4  26]
 [ 27 118  27   6  22]
 [ 43  23  80  17  37]
 [ 14  15  33 105  33]
 [ 13  18  40  12 117]]


In [15]:
# ==============================
# Step 7 — Save Models for Later Use
# ==============================
os.makedirs("models", exist_ok=True)
ft_model.save("models/fine_tuned_resnet50_SVM.h5")
joblib.dump(scaler, "models/scaler_SVM.pkl")
joblib.dump(pca, "models/pca_SVM.pkl")
joblib.dump(clf, "models/svm_face_shape.pkl")
print("Models saved successfully!")

Models saved successfully!


In [11]:
# ==============================
# Step 8 — Hairstyle Recommendation System
# ==============================

def recommend_hairstyle(face_shape):
    recommendations = {
        "Heart": [
            "Side-swept bangs",
            "Chin-length bobs",
            "Soft layers around the chin"
        ],
        "Oblong": [
            "Wavy styles with volume",
            "Curtain bangs",
            "Shoulder-length cuts"
        ],
        "Oval": [
            "Almost any style suits",
            "Long waves",
            "Pixie cuts"
        ],
        "Round": [
            "Long layers to elongate the face",
            "Side parts",
            "High ponytails"
        ],
        "Square": [
            "Soft curls or waves",
            "Layered cuts that soften the jawline",
            "Side-swept fringes"
        ]
    }

    return recommendations.get(face_shape, ["Try consulting a stylist!"])

In [13]:
# ==============================
# Step 9 — Predict Face Shape for User Image
# ==============================

from tensorflow.keras.preprocessing import image

def predict_face_shape(img_path):
    # Load and preprocess image
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)

    # Extract features using trained ResNet50
    features = feature_extractor.predict(img_array, verbose=0)
    features = features.reshape(features.shape[0], -1)

    # Scale and transform features
    features_scaled = scaler.transform(features)
    features_pca = pca.transform(features_scaled)

    # Predict with SVM
    pred = clf.predict(features_pca)[0]
    labels = ['Heart', 'Oblong', 'Oval', 'Round', 'Square']
    predicted_shape = labels[int(pred)]
    return predicted_shape


In [14]:
# ==============================
# Step 10 — Demo: Predict + Recommend
# ==============================

test_img_path = "../../data sheets/zendaya.jpg"  # replace with your test image
if os.path.exists(test_img_path):
    face_shape = predict_face_shape(test_img_path)
    hairstyles = recommend_hairstyle(face_shape)
    print("\nPredicted Face Shape:", face_shape)
    print("Recommended Hairstyles:")
    for s in hairstyles:
        print("-", s)
else:
    print("\nUpload an image at 'uploads/test_face.jpg' to test predictions.")


Predicted Face Shape: Heart
Recommended Hairstyles:
- Side-swept bangs
- Chin-length bobs
- Soft layers around the chin
